Hier werden nun alle ICON-D2 Daten mit den 13 Uhr Traffic Daten gemerged.
Basierend auf die Standorte der Sensoren werden die korrekten Werte für Wind direction, speed, temp und prec zugewiesen.


In [1]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np

# =========================================================
# 1. Pfade definieren
# =========================================================
verkehr_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exponent2_13Uhr_Verkehrs_Proxys_Komplett.csv"
sensor_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\sensor_community_mit_kopfklinik_may_august.csv"
raster_base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\New_ICON_D2_Rasters"

# Ausgabedatei
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_Verkehr_und_Wetter.csv"

# =========================================================
# 2. Sensor-Koordinaten extrahieren und bereinigen
# =========================================================
print("Lade Sensor-Daten und extrahiere Koordinaten...")
df_sensors = pd.read_csv(sensor_file, sep=";", decimal=",")

# Kommas durch Punkte ersetzen und in Zahlen umwandeln
for col in ["lat", "lon"]:
    df_sensors[col] = df_sensors[col].astype(str).str.replace(",", ".", regex=False)
    df_sensors[col] = pd.to_numeric(df_sensors[col], errors="coerce")

# Nur eindeutige Stationen behalten
unique_sensors = df_sensors[["sensor_id", "lat", "lon"]].drop_duplicates(subset=["sensor_id"]).dropna()
unique_sensors["station_id"] = unique_sensors["sensor_id"].astype(float).astype(int).astype(str)
unique_sensors = unique_sensors[["station_id", "lat", "lon"]]

print(f"-> {len(unique_sensors)} eindeutige Sensor-Standorte gefunden.")

# =========================================================
# 3. Verkehrsdaten laden und mit Koordinaten mergen
# =========================================================
print("Lade Verkehrs-Proxys und füge Koordinaten hinzu...")
df_traffic = pd.read_csv(verkehr_file, sep=";", decimal=",")
df_traffic["station_id"] = df_traffic["station_id"].astype(str)

# Merge: Sensorkoordinaten an die Verkehrsdaten hängen
df_merged = pd.merge(df_traffic, unique_sensors, on="station_id", how="left")

# =========================================================
# 4. Koordinatensystem für die Raster-Abfrage anpassen
# =========================================================
print("Transformiere Koordinaten in das metrische System (EPSG:25832)...")
# GeoDataFrame aus den Koordinaten erstellen (WGS84 = EPSG:4326)
gdf = gpd.GeoDataFrame(
    df_merged,
    geometry=gpd.points_from_xy(df_merged["lon"], df_merged["lat"]),
    crs="EPSG:4326"
)

# In das System der GeoTIFFs umwandeln
gdf = gdf.to_crs("EPSG:25832")
# X- und Y-Koordinaten als Liste von Tupeln für rasterio vorbereiten
df_merged["x_utm"] = gdf.geometry.x
df_merged["y_utm"] = gdf.geometry.y

# =========================================================
# 5. Wetter-Daten aus den Rastern extrahieren
# =========================================================
print("Extrahiere Wetterdaten aus den ICON-D2 GeoTIFFs...")

# Neue Spalten initialisieren
df_merged["icon_temp_2m"] = np.nan
df_merged["icon_wind_speed"] = np.nan
df_merged["icon_wind_direction"] = np.nan
df_merged["icon_precip_12h"] = np.nan
df_merged["icon_relative_humidity"] = np.nan

# Mapping der Raster-Ordner und Dateinamen-Präfixe
raster_mappings = {
    "icon_temp_2m": {"folder": "Temp_2m", "prefix": "ICON_D2_Temp_"},
    "icon_wind_speed": {"folder": "Wind_Speed", "prefix": "ICON_D2_WindSpeed_"},
    "icon_wind_direction": {"folder": "Wind_Direction", "prefix": "wind_dir_"}, # Passe Präfix ggf. an! wind_dir_2024-05-01
    "icon_precip_12h": {"folder": "Precipitation_12h", "prefix": "ICON_D2_Precip_12h_"},
    "icon_relative_humidity":{"folder": "Relative_Humidity_2m", "prefix": "ICON_D2_RelHumid_"}
}

unique_dates = df_merged["date"].unique()

for current_date in unique_dates:
    # Filter für den aktuellen Tag
    mask = df_merged["date"] == current_date
    day_points = df_merged.loc[mask, ["x_utm", "y_utm"]].values

    # Als Liste von Tupeln formatieren: [(x1, y1), (x2, y2), ...]
    coords_list = [(x, y) for x, y in day_points]

    for col_name, mapping in raster_mappings.items():
        # Dateipfad zusammenbauen
        tif_filename = f"{mapping['prefix']}{current_date}.tif"
        tif_path = os.path.join(raster_base_dir, mapping["folder"], tif_filename)

        # Prüfen, ob die Datei existiert (verhindert Absturz bei fehlenden Tagen)
        if os.path.exists(tif_path):
            with rasterio.open(tif_path) as src:
                # rasterio.sample extrahiert die Werte an den Koordinaten
                sampled_values = [val[0] for val in src.sample(coords_list)]
                # Werte in den DataFrame eintragen
                df_merged.loc[mask, col_name] = sampled_values
        else:
            print(f"Warnung: Fehlende Datei {tif_filename}")

# =========================================================
# 6. Aufräumen und Speichern
# =========================================================
# Unnötige Hilfsspalten entfernen
df_final = df_merged.drop(columns=["x_utm", "y_utm"])

df_final.to_csv(output_file, index=False, sep=";", decimal=",")

print("-" * 60)
print(f"ERFOLG! Der Datensatz mit Verkehr UND Wetter liegt hier:\n{output_file}")
print("-" * 60)

Lade Sensor-Daten und extrahiere Koordinaten...
-> 9 eindeutige Sensor-Standorte gefunden.
Lade Verkehrs-Proxys und füge Koordinaten hinzu...
Transformiere Koordinaten in das metrische System (EPSG:25832)...
Extrahiere Wetterdaten aus den ICON-D2 GeoTIFFs...
------------------------------------------------------------
ERFOLG! Der Datensatz mit Verkehr UND Wetter liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_Verkehr_und_Wetter.csv
------------------------------------------------------------
